<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z337_TendenciaClasica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tendencia Clásica — Spearman, ANOVA en bloques, Levene

Tres enfoques clásicos para detectar y seguir tendencia:

**2. Spearman(tn, tiempo)**
Correlación de Spearman entre el valor y el índice temporal sobre los últimos N meses.
- ρ > umbral con p < 0.05 → tendencia real → extrapolar con Sen's Slope
- Si no → mediana reciente

**3. ANOVA en bloques temporales**
Divide la historia en 3 bloques (pasado / medio / reciente).
- Si las medias son distintas (p < 0.05) → hubo cambio de nivel → usar solo el bloque reciente
- Si no → usar toda la historia (mediana global)

**4. Levene en bloques**
Igual que ANOVA pero testea si la **varianza** cambia entre bloques.
- Si la varianza cambió → el producto está en un régimen nuevo → anclar al bloque reciente
- Si no → historia estable → mediana global

Los tres en backtesting + submit con todas las variantes.

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell
mkdir -p "/content/.drive/My Drive/labo3" /content/buckets
ln -sfn "/content/.drive/My Drive/labo3" /content/buckets/b1
mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json
mkdir -p /content/buckets/b1/datasets /content/datasets
descargar() {
  d="/content/buckets/b1/datasets/"
  u="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  if ! test -f "$d$1"; then wget "$u$1" -O "$d$1"; fi
  if ! test -f "/content/datasets/$1"; then cp "$d$1" "/content/datasets/$1"; fi
}
descargar sell-in.txt.gz
descargar product_id_apredecir201912.txt

In [ ]:
!pip install uv -q && uv pip install -q kaggle

In [ ]:
import os
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import spearmanr, f_oneway, levene, chi2
import warnings
warnings.filterwarnings('ignore')

COMPETENCIA  = 'labo-iii-2026-rosario'
PERIODO_CORTE  = 201910
PERIODO_TARGET = 201912

dataset      = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator='\t')
tb_ventas    = dataset.group_by('product_id','periodo').agg(pl.col('tn').sum()).sort(['product_id','periodo'])
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator='\t')
tb_ventas    = tb_ventas.join(tb_apredecir, on='product_id', how='inner').sort(['product_id','periodo'])
tb_train     = tb_ventas.filter(pl.col('periodo') <= PERIODO_CORTE)
tb_real      = (tb_ventas.filter(pl.col('periodo') == PERIODO_TARGET)
                .select(['product_id','tn']).rename({'tn':'tn_real'}))
productos    = tb_apredecir['product_id'].to_list()
print(f'{len(productos)} productos')

# Predictores

### Sen's Slope
Pendiente robusta: mediana de todas las pendientes par a par `(y_j - y_i) / (j - i)`.
No la rompe un mes anómalo. Se usa para extrapolar cuando se detecta tendencia.

In [ ]:
def sens_slope(serie):
    """Pendiente de Sen: mediana de todas las pendientes par a par."""
    n = len(serie)
    slopes = []
    for i in range(n):
        for j in range(i+1, n):
            slopes.append((serie[j] - serie[i]) / (j - i))
    return float(np.median(slopes))


def extrapolar_sens(serie, ventana, horizonte=2):
    """Extrapola usando Sen's Slope sobre los últimos `ventana` meses."""
    w = min(ventana, len(serie))
    s = serie[-w:]
    slope = sens_slope(s)
    # último valor + slope * horizonte
    pred  = s[-1] + slope * horizonte
    return max(float(pred), 0.0)


# ── 2. Spearman ───────────────────────────────────────────────────────────────
def pred_spearman(serie, ventana, alpha=0.05, horizonte=2):
    """
    Calcula Spearman(tn, tiempo) sobre los últimos `ventana` meses.
    Si hay tendencia significativa (p < alpha) → extrapola con Sen's Slope.
    Si no → mediana reciente.
    """
    w = min(ventana, len(serie))
    s = serie[-w:]
    if w < 4:
        return max(float(np.median(s)), 0.0)
    rho, pval = spearmanr(np.arange(w), s)
    if pval < alpha:
        return extrapolar_sens(serie, ventana, horizonte)
    return max(float(np.median(s)), 0.0)


# ── 3. ANOVA en bloques ───────────────────────────────────────────────────────
def pred_anova_bloques(serie, n_bloques=3, alpha=0.05):
    """
    Divide la historia en `n_bloques` iguales.
    ANOVA testea si las medias entre bloques son distintas.
    Si p < alpha (hubo cambio de nivel) → usar solo el bloque más reciente.
    Si no → mediana de toda la historia.
    """
    n = len(serie)
    if n < n_bloques * 2:
        return max(float(np.median(serie)), 0.0)

    tam = n // n_bloques
    bloques = [serie[i*tam:(i+1)*tam] for i in range(n_bloques)]
    # el último bloque toma el resto
    bloques[-1] = serie[(n_bloques-1)*tam:]

    _, pval = f_oneway(*bloques)
    if pval < alpha:
        return max(float(np.median(bloques[-1])), 0.0)
    return max(float(np.median(serie)), 0.0)


# ── 4. Levene en bloques ──────────────────────────────────────────────────────
def pred_levene_bloques(serie, n_bloques=3, alpha=0.05):
    """
    Divide la historia en `n_bloques` iguales.
    Levene testea si la varianza cambió entre bloques.
    Si p < alpha (régimen de varianza nuevo) → anclar al bloque reciente.
    Si no → mediana global.
    """
    n = len(serie)
    if n < n_bloques * 2:
        return max(float(np.median(serie)), 0.0)

    tam = n // n_bloques
    bloques = [serie[i*tam:(i+1)*tam] for i in range(n_bloques)]
    bloques[-1] = serie[(n_bloques-1)*tam:]

    try:
        _, pval = levene(*bloques)
    except Exception:
        return max(float(np.median(serie)), 0.0)

    if pval < alpha:
        return max(float(np.median(bloques[-1])), 0.0)
    return max(float(np.median(serie)), 0.0)


print('Funciones OK')

# Backtesting

In [ ]:
ventanas   = [6, 12, 18, 24]
n_bloques  = [2, 3, 4]
alphas     = [0.05, 0.10]

resultados = {pid: {} for pid in productos}

for pid in productos:
    serie = (
        tb_train.filter(pl.col('product_id') == pid)
        .sort('periodo')['tn'].to_numpy().astype(float)
    )
    resultados[pid]['tn_real'] = None  # se completa después

    # naive baseline
    resultados[pid]['naive'] = max(float(np.median(serie[-6:])), 0.0)

    # Spearman
    for v in ventanas:
        for a in alphas:
            resultados[pid][f'spearman_v{v}_a{int(a*100)}'] = pred_spearman(serie, v, alpha=a)

    # ANOVA bloques
    for nb in n_bloques:
        for a in alphas:
            resultados[pid][f'anova_b{nb}_a{int(a*100)}'] = pred_anova_bloques(serie, nb, alpha=a)

    # Levene bloques
    for nb in n_bloques:
        for a in alphas:
            resultados[pid][f'levene_b{nb}_a{int(a*100)}'] = pred_levene_bloques(serie, nb, alpha=a)

# join con reales
tb_preds = pl.DataFrame([{'product_id': pid, **v} for pid, v in resultados.items()]).drop('tn_real')
tb_bt    = tb_real.join(tb_preds, on='product_id', how='left')

modelos = [c for c in tb_bt.columns if c not in ('product_id', 'tn_real')]
for m in modelos:
    tb_bt = tb_bt.with_columns(
        (pl.col('tn_real') - pl.col(m)).abs().alias(f'err_{m}')
    )

print('Backtesting listo')

In [ ]:
rmse_dict = {}
for m in modelos:
    rmse_dict[m] = float(np.sqrt((tb_bt[f'err_{m}'] ** 2).mean()))

rmse_naive = rmse_dict['naive']

# ordenar por RMSE
ranking = sorted(rmse_dict.items(), key=lambda x: x[1])

print(f"{'modelo':35s}  RMSE    vs naive")
print('-' * 55)
for m, rmse in ranking:
    delta = rmse - rmse_naive
    marca = ' ◄ MEJOR' if m == ranking[0][0] else ''
    tag   = '(baseline)' if m == 'naive' else f'{delta:+.4f}'
    print(f"  {m:33s}: {rmse:.4f}  {tag}{marca}")

# ¿Cuántos productos detecta tendencia cada método?

In [ ]:
conteos = {'spearman': {}, 'anova': {}, 'levene': {}}

for pid in productos:
    serie = (
        tb_train.filter(pl.col('product_id') == pid)
        .sort('periodo')['tn'].to_numpy().astype(float)
    )
    for v in ventanas:
        for a in alphas:
            key = f'v{v}_a{int(a*100)}'
            w = min(v, len(serie))
            s = serie[-w:]
            if w >= 4:
                _, pval = spearmanr(np.arange(w), s)
                conteos['spearman'][key] = conteos['spearman'].get(key, 0) + (1 if pval < a else 0)

    for nb in n_bloques:
        for a in alphas:
            key = f'b{nb}_a{int(a*100)}'
            n = len(serie)
            if n >= nb * 2:
                tam = n // nb
                bloques = [serie[i*tam:(i+1)*tam] for i in range(nb)]
                bloques[-1] = serie[(nb-1)*tam:]
                _, pval_a = f_oneway(*bloques)
                conteos['anova'][key]  = conteos['anova'].get(key, 0)  + (1 if pval_a < a else 0)
                try:
                    _, pval_l = levene(*bloques)
                    conteos['levene'][key] = conteos['levene'].get(key, 0) + (1 if pval_l < a else 0)
                except Exception:
                    pass

print(f"{'método':30s}  n detectados  %")
print('-' * 50)
for metodo, d in conteos.items():
    for k, n in sorted(d.items()):
        print(f"  {metodo}_{k:20s}: {n:4d}  ({100*n/len(productos):.1f}%)")
    print()

# Visualización — casos donde Spearman activa y Sen's Slope gana

In [ ]:
# mejor variante de spearman según backtesting
mejor_spearman = min(
    {k: v for k, v in rmse_dict.items() if k.startswith('spearman')}.items(),
    key=lambda x: x[1]
)[0]
print(f'Mejor variante Spearman: {mejor_spearman}  RMSE={rmse_dict[mejor_spearman]:.4f}')

tb_bt = tb_bt.with_columns(
    (pl.col('naive') - pl.col(mejor_spearman)).alias('mejora_spearman')
)

top6 = tb_bt.sort('mejora_spearman', descending=True).head(6)['product_id'].to_list()

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, pid in enumerate(top6):
    serie_full = tb_ventas.filter(pl.col('product_id') == pid).sort('periodo')
    periodos_  = serie_full['periodo'].to_list()
    tn_        = serie_full['tn'].to_numpy().astype(float)
    idx_corte  = next((j for j, p in enumerate(periodos_) if p > PERIODO_CORTE), len(periodos_))
    idx_target = next((j for j, p in enumerate(periodos_) if p == PERIODO_TARGET), None)
    tn_train_  = tn_[:idx_corte]

    row       = tb_bt.filter(pl.col('product_id') == pid)
    real_val  = float(row['tn_real'][0])
    naive_val = float(row['naive'][0])
    spear_val = float(row[mejor_spearman][0])
    mejora    = float(row['mejora_spearman'][0])

    ax = axes[i]
    ax.plot(range(len(tn_train_)), tn_train_, 'o-', color='steelblue',
            markersize=3, linewidth=1.5)

    if idx_target is not None:
        t = idx_target
        ax.scatter([t], [real_val],  color='black',  s=90, zorder=6, label=f'real={real_val:.1f}')
        ax.scatter([t], [spear_val], color='tomato', s=60, zorder=5, marker='D', label=f'spearman={spear_val:.1f}')
        ax.scatter([t], [naive_val], color='gray',   s=40, zorder=5, marker='D', label=f'naive={naive_val:.1f}')

    ax.set_title(f'pid {pid}  mejora={mejora:.1f}', fontsize=8)
    ax.legend(fontsize=6)

fig.suptitle(f'Casos donde {mejor_spearman} gana al naive', fontsize=10)
plt.tight_layout()
plt.show()

# Submit — todas las variantes ganadoras

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
    os.system(f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"')

# precalcular series con datos completos (hasta 201912)
series_full = {
    pid: tb_ventas.filter(pl.col('product_id') == pid)
         .sort('periodo')['tn'].to_numpy().astype(float)
    for pid in productos
}

# submitear todas las variantes que superaron al naive en backtesting
mejores = [(m, rmse) for m, rmse in rmse_dict.items()
           if rmse < rmse_naive and m != 'naive']
mejores.sort(key=lambda x: x[1])

print(f'{len(mejores)} variantes mejores que naive → submiteando...')
print()

for nombre, rmse_bt in mejores:
    preds = []
    for pid in productos:
        s = series_full[pid]
        # reconstruir la llamada según el prefijo del nombre
        if nombre.startswith('spearman'):
            partes = nombre.split('_')
            v = int(partes[1][1:])
            a = int(partes[2][1:]) / 100
            pred = pred_spearman(s, v, alpha=a)
        elif nombre.startswith('anova'):
            partes = nombre.split('_')
            nb = int(partes[1][1:])
            a  = int(partes[2][1:]) / 100
            pred = pred_anova_bloques(s, nb, alpha=a)
        elif nombre.startswith('levene'):
            partes = nombre.split('_')
            nb = int(partes[1][1:])
            a  = int(partes[2][1:]) / 100
            pred = pred_levene_bloques(s, nb, alpha=a)
        else:
            pred = max(float(np.median(s[-6:])), 0.0)
        preds.append({'product_id': pid, 'tn': pred})

    archivo = f"{nombre}.csv"
    pl.DataFrame(preds).write_csv(archivo)
    kaggle_submit(COMPETENCIA, archivo, f'{nombre} RMSE_bt={rmse_bt:.4f}')
    print(f'  submitted: {nombre}  (RMSE_bt={rmse_bt:.4f})')